In [1]:
from rdflib import Graph, RDF, SDO, URIRef, BNode

In [2]:
def print_stats(g):
    # Measuring graph size:
    print("Amount of triples: ", len(g))

    entities = set(g.subjects(RDF.type, None))
    print(f"Total Unique Entities: {len(entities)}")

    query_products = """
    PREFIX schema: <https://schema.org/>

    SELECT (COUNT(DISTINCT ?s) AS ?count)
    WHERE {
        ?s a schema:Product .
    }
    """

    result = g.query(query_products)
    for row in result:
        print(f"Total Product Entities: {row['count']}")

    num_edges = len(g)

    # Find amount of unmatched properties
    print("Unclassified properties:", len(list(g.triples((None, SDO.additionalProperty, None)))))


    # Measuring connectivity:
    # We collect all subjects and all objects to get every unique node
    nodes = set()
    for s, p, o in g:
        nodes.add(s)
        nodes.add(o)

    num_nodes = len(nodes)

    if num_nodes == 0:
        return 0

    avg_degree = (2 * num_edges) / num_nodes

    # Often, we don't want to count Literals (strings/numbers) as "nodes"
    entities = {n for n in nodes if isinstance(n, (URIRef, BNode))}
    num_entities = len(entities)

    # Triples where both subject and object are entities
    # This represents internal graph connectivity
    internal_edges = 0
    for s, p, o in g:
        if isinstance(s, (URIRef, BNode)) and isinstance(o, (URIRef, BNode)):
            internal_edges += 1


    avg_entity_degree = (2 * internal_edges) / num_entities if num_entities > 0 else 0

    print(f"--- Connectivity Stats ---")
    print(f"Total Nodes (V):      {num_nodes}")
    print(f"Total Edges (E):      {num_edges}")
    print(f"Average Degree:       {avg_degree:.2f}")
    print(f"Avg Entity Degree:    {avg_entity_degree:.2f}")


In [3]:
g = Graph().parse("./graphs/graph_1000.ttl")

print("Stats for small graph:")
print_stats(g)

Stats for small graph:
Amount of triples:  55744
Total Unique Entities: 10426
Total Product Entities: 2000
Unclassified properties: 6141
--- Connectivity Stats ---
Total Nodes (V):      25190
Total Edges (E):      55744
Average Degree:       4.43
Avg Entity Degree:    3.84


In [4]:
g = Graph().parse("./graphs/graph_5000.ttl")

print("Stats for medium graph:")
print_stats(g)

Stats for medium graph:
Amount of triples:  275289
Total Unique Entities: 51471
Total Product Entities: 10000
Unclassified properties: 30714
--- Connectivity Stats ---
Total Nodes (V):      115427
Total Edges (E):      275289
Average Degree:       4.77
Avg Entity Degree:    3.84


In [5]:
g = Graph().parse("./graphs/graph_20000.ttl")

print("Stats for large graph:")
print_stats(g)

Stats for large graph:
Amount of triples:  1216717
Total Unique Entities: 230114
Total Product Entities: 40533
Unclassified properties: 147447
--- Connectivity Stats ---
Total Nodes (V):      466313
Total Edges (E):      1216717
Average Degree:       5.22
Avg Entity Degree:    3.86
